# ALPR Colombia - Entrenar YOLOv8 con persistencia en Drive
Guarda checkpoints a Google Drive cada época. Si la sesión se cierra,
vuelve a montar Drive y reanuda desde el último checkpoint.

In [ ]:
# 0. Montar Google Drive para persistencia
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/alpr_colombia_checkpoints'
import os, json, time, shutil, glob
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Checkpoints en: {DRIVE_DIR}')

In [ ]:
# 1. Verificar GPU
import torch
print(f'GPU disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    !nvidia-smi --query-gpu=memory.total,memory.free --format=csv,noheader

In [ ]:
# 2. Instalar dependencias
!pip install -q ultralytics roboflow

---
## Configuración
Elige qué datasets usar y los parámetros de entrenamiento.

In [ ]:
# --- DATASETS ---
USE_NUEVO_1 = True   # alo-fwpow/placas-gshtx v4
USE_NUEVO_2 = True   # placasp/placas-bpbge    v2
USE_NUEVO_3 = True   # usco-thj9e/placas-colombia-ixdpr v5
USE_ORIGINAL = False # placas-colombia/proyecto-placas v2

# --- ENTRENAMIENTO ---
EPOCHS = 100
BATCH  = 16
IMGSZ  = 640

print('Configuración lista.')

In [ ]:
# 3. Descargar datasets desde Roboflow
from roboflow import Roboflow

rf = Roboflow(api_key='bUpcZuNnrkKdqieWD0qb')

datasets = []
if USE_NUEVO_1:
    print('Descargando placas-gshtx v4...')
    d1 = rf.workspace('alo-fwpow').project('placas-gshtx').version(4).download('yolov8')
    datasets.append(('gshtx', d1.location))
if USE_NUEVO_2:
    print('Descargando placas-bpbge v2...')
    d2 = rf.workspace('placasp').project('placas-bpbge').version(2).download('yolov8')
    datasets.append(('bpbge', d2.location))
if USE_NUEVO_3:
    print('Descargando placas-colombia-ixdpr v5...')
    d3 = rf.workspace('usco-thj9e').project('placas-colombia-ixdpr').version(5).download('yolov8')
    datasets.append(('ixdpr', d3.location))
if USE_ORIGINAL:
    print('Descargando proyecto-placas v2...')
    d0 = rf.workspace('placas-colombia').project('proyecto-placas-8arfj').version(2).download('yolov8')
    datasets.append(('original', d0.location))

print(f'\nTotal datasets descargados: {len(datasets)}')

In [ ]:
# 4. Fusionar datasets en una sola estructura
COMBINED = '/content/placas-combinado'
os.makedirs(COMBINED, exist_ok=True)

total_images = 0
for prefix, loc in datasets:
    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(loc, split, 'images')
        lbl_dir = os.path.join(loc, split, 'labels')
        if not os.path.isdir(img_dir):
            continue

        dst_img = os.path.join(COMBINED, split, 'images')
        dst_lbl = os.path.join(COMBINED, split, 'labels')
        os.makedirs(dst_img, exist_ok=True)
        os.makedirs(dst_lbl, exist_ok=True)

        imgs = glob.glob(os.path.join(img_dir, '*'))
        for src in imgs:
            fname = os.path.basename(src)
            new_name = f'{prefix}_{fname}'
            shutil.copy2(src, os.path.join(dst_img, new_name))

            base, ext = os.path.splitext(fname)
            lbl_src = os.path.join(lbl_dir, base + '.txt')
            if os.path.exists(lbl_src):
                shutil.copy2(lbl_src, os.path.join(dst_lbl, f'{prefix}_{base}.txt'))

        count = len(imgs)
        total_images += count
        print(f'{prefix}/{split}: {count} imagenes')

print(f'\nTotal imagenes combinadas: {total_images}')
for split in ['train', 'valid', 'test']:
    p = os.path.join(COMBINED, split, 'images')
    if os.path.isdir(p):
        print(f'  {split}: {len(os.listdir(p))} imagenes')

In [ ]:
# 5. Crear data.yaml combinado
import yaml

data_yaml = {
    'train': os.path.join(COMBINED, 'train', 'images'),
    'val':   os.path.join(COMBINED, 'valid', 'images'),
    'test':  os.path.join(COMBINED, 'test', 'images') if os.path.isdir(os.path.join(COMBINED, 'test', 'images')) else '',
    'nc': 1,
    'names': ['placa'],
}

yaml_path = os.path.join(COMBINED, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f'data.yaml creado en: {yaml_path}')
!cat {yaml_path}

In [ ]:
# 6. Cargar o reanudar modelo
from ultralytics import YOLO

RESUME_PATH = os.path.join(DRIVE_DIR, 'last.pt')
if os.path.exists(RESUME_PATH):
    print(f'[DRIVE] Reanudando desde checkpoint existente: {RESUME_PATH}')
    model = YOLO(RESUME_PATH)
else:
    print('[DRIVE] No hay checkpoint previo. Iniciando desde base...')
    model = YOLO('yolov8n.pt')

print('Modelo cargado.')

In [ ]:
# 7. Callback para guardar checkpoint a Drive cada época
from ultralytics.models.yolo.detect.train import DetectionTrainer

def save_to_drive(trainer):
    epoch = trainer.epoch
    best_pt = os.path.join(trainer.save_dir, 'weights', 'best.pt')
    last_pt = os.path.join(trainer.save_dir, 'weights', 'last.pt')

    if os.path.exists(best_pt):
        shutil.copy2(best_pt, os.path.join(DRIVE_DIR, 'best.pt'))
        shutil.copy2(best_pt, os.path.join(DRIVE_DIR, f'best_epoch_{epoch:03d}.pt'))
    if os.path.exists(last_pt):
        shutil.copy2(last_pt, os.path.join(DRIVE_DIR, 'last.pt'))

    # Guardar progreso
    progress = {
        'epoch': epoch,
        'timestamp': time.time(),
        'date': time.strftime('%Y-%m-%d %H:%M:%S'),
    }
    if hasattr(trainer, 'metrics') and trainer.metrics:
        progress['map50'] = round(float(trainer.metrics.get('metrics/mAP50(B)', 0)), 4)
        progress['map50_95'] = round(float(trainer.metrics.get('metrics/mAP50-95(B)', 0)), 4)
        progress['precision'] = round(float(trainer.metrics.get('metrics/precision(B)', 0)), 4)
        progress['recall'] = round(float(trainer.metrics.get('metrics/recall(B)', 0)), 4)
    with open(os.path.join(DRIVE_DIR, 'progress.json'), 'w') as f:
        json.dump(progress, f)

    print(f'[DRIVE] Checkpoint época {epoch} guardado.')


# Registrar callback on_fit_epoch_end
# DetectionTrainer llama on_fit_epoch_end al final de cada época
from ultralytics.utils.callbacks import add_integration_callbacks

# Si ya hay checkpoints reanudados, reanudar desde la última época
start_epoch = 0
progress_path = os.path.join(DRIVE_DIR, 'progress.json')
if os.path.exists(progress_path):
    with open(progress_path) as f:
        prog = json.load(f)
    start_epoch = prog.get('epoch', 0) + 1
    # Eliminar épocas futuras del historial de YOLO
    print(f'[DRIVE] Reanudando desde época {start_epoch + 1}/{EPOCHS}')

print(f'Callback registrado.')

In [ ]:
# 8. Entrenar YOLOv8n con checkpointing a Drive
from ultralytics import YOLO

# Metodo manual con entrenamiento por lotes de epocas para tener control
# Entrenar en bloques de 5 epocas, guardando a Drive cada vez
# Recargar modelo desde last.pt al inicio de cada lote para evitar
# que model.train() corrompa el estado interno en iteraciones siguientes
EPOCHS_REMAINING = EPOCHS - start_epoch
BATCH_SIZE = 5  # epocas por lote
LAST_PT = os.path.join(DRIVE_DIR, 'last.pt')

for batch_start in range(start_epoch, EPOCHS, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, EPOCHS)
    n_epochs = batch_end - batch_start
    print(f'\n{"="*50}')
    print(f'  Epocas {batch_start+1}-{batch_end} de {EPOCHS}')
    print(f'{"="*50}\n')

    # Cargar modelo fresco cada iteracion
    if os.path.exists(LAST_PT):
        model = YOLO(LAST_PT)
        print(f'[DRIVE] Cargado last.pt para continuar entrenamiento')
    else:
        model = YOLO('yolov8n.pt')
        print(f'[DRIVE] Iniciando desde base')

    results = model.train(
        data=yaml_path,
        epochs=n_epochs,
        imgsz=IMGSZ,
        batch=BATCH,
        name='plate_detector',
        patience=30,
        save=True,
        plots=True,
        exist_ok=True,
        resume=False,
        # Aumentacion de datos
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=15.0,
        translate=0.1,
        scale=0.3,
        fliplr=0.5,
        flipud=0.1,
        mosaic=1.0,
        mixup=0.1,
        # Optimizador
        optimizer='AdamW',
        lr0=0.001,
        weight_decay=0.0005,
        warmup_epochs=3,
        cos_lr=True,
        # Recursos
        workers=4,
        device='cuda' if torch.cuda.is_available() else 'cpu',
    )

    # Guardar a Drive despues del lote
    save_dir = results.save_dir
    best_pt = os.path.join(save_dir, 'weights', 'best.pt')
    last_pt = os.path.join(save_dir, 'weights', 'last.pt')

    if os.path.exists(best_pt):
        shutil.copy2(best_pt, os.path.join(DRIVE_DIR, 'best.pt'))
        shutil.copy2(best_pt, os.path.join(DRIVE_DIR, f'best_epoch_{batch_end:03d}.pt'))
    if os.path.exists(last_pt):
        shutil.copy2(last_pt, os.path.join(DRIVE_DIR, 'last.pt'))

    # Extraer metricas
    maps = results.results_dict
    progress = {
        'epoch': batch_end,
        'timestamp': time.time(),
        'date': time.strftime('%Y-%m-%d %H:%M:%S'),
        'map50': round(float(maps.get('metrics/mAP50(B)', 0)), 4),
        'map50_95': round(float(maps.get('metrics/mAP50-95(B)', 0)), 4),
        'precision': round(float(maps.get('metrics/precision(B)', 0)), 4),
        'recall': round(float(maps.get('metrics/recall(B)', 0)), 4),
    }
    with open(os.path.join(DRIVE_DIR, 'progress.json'), 'w') as f:
        json.dump(progress, f)

    print(f'[DRIVE] Lote {batch_start+1}-{batch_end} guardado. mAP50: {progress["map50"]:.3f}')

print(f'\nEntrenamiento completado: {EPOCHS} epocas.')


In [ ]:
# 9. Evaluar modelo final
# Recargar el mejor modelo guardado
from ultralytics import YOLO

final_model = YOLO(os.path.join(DRIVE_DIR, 'best.pt'))
metrics = final_model.val(data=yaml_path)
print(f'\nResultados finales:')
print(f'  mAP50:    {metrics.box.map50:.3f}')
print(f'  mAP50-95: {metrics.box.map:.3f}')
print(f'  Precision: {metrics.box.mp:.3f}')
print(f'  Recall:   {metrics.box.mr:.3f}')

In [ ]:
# 10. Compartir enlace de Drive para descarga automática
from google.colab import files
import subprocess

# Obtener ID de la carpeta de Drive
# La carpeta se creó al inicio en /content/drive/MyDrive/alpr_colombia_checkpoints
# En Drive, el ID se obtiene abriendo la carpeta y copiando de la URL
# O con el comando drive:
print('=' * 60)
print('INSTRUCCIONES PARA CONFIGURAR LA DESCARGA AUTOMATICA')
print('=' * 60)
print(f'\n1. Abre Google Drive y ve a: Mi Unidad/alpr_colombia_checkpoints/')
print('2. Abre la carpeta. La URL se ve as\u00ed:')
print('   https://drive.google.com/drive/folders/ABC123XXX')
print('   Copia el ID despu\u00e9s de "folders/"')
print('3. Pon ese ID en watch_checkpoint.py (DRIVE_FOLDER_ID)')
print('4. Aseg\u00farate que la carpeta tenga acceso: "Cualquier persona con el enlace"\n')
print(f'\nArchivos en Drive:')
for f in sorted(os.listdir(DRIVE_DIR)):
    size = os.path.getsize(os.path.join(DRIVE_DIR, f)) / 1e6
    print(f'  {f:30s} {size:.1f} MB')

# Compartir carpeta
print(f'\nComparte la carpeta desde Drive > clic derecho > Compartir > ')
print(f'"Cualquier persona con el enlace puede ver"')
print(f'\nO ejecuta esto en una celda para obtener el enlace:')

In [ ]:
# (Opcional) Obtener enlace de Drive
# Necesita autorizaci\u00f3n de Google API
try:
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload
    auth.authenticate_user()

    drive_service = build('drive', 'v3')

    # Buscar la carpeta
    results = drive_service.files().list(
        q="name='alpr_colombia_checkpoints' and mimeType='application/vnd.google-apps.folder' and 'root' in parents",
        fields='files(id, webViewLink)'
    ).execute()

    items = results.get('files', [])
    if items:
        folder_id = items[0]['id']
        link = items[0]['webViewLink']
        print(f'Carpeta ID: {folder_id}')
        print(f'Enlace: {link}')
        print(f'\nEl FOLDER_ID para watch_checkpoint.py es: {folder_id}')
    else:
        print('No se encontr\u00f3 la carpeta. Cr\u00e9ala manualmente en Drive.')
except Exception as e:
    print(f'No se pudo obtener el ID autom\u00e1ticamente: {e}')
    print('Copia el ID manualmente desde la URL de Drive.')

In [ ]:
# 11. Exportar modelo final
import shutil

# Copia final a content
if os.path.exists(os.path.join(DRIVE_DIR, 'best.pt')):
    shutil.copy2(os.path.join(DRIVE_DIR, 'best.pt'), '/content/plate_detector.pt')
    print('Modelo final copiado a /content/plate_detector.pt')

# Descargar manual
from google.colab import files
print('\nClick para descargar:')
files.download('/content/plate_detector.pt')